# Predict Demo

Load the SVM that was trained in `train_model.ipynb` and run it on a single image.

Change the `test_path` in the last cell to whichever image you want to try.

## 1. Imports

In [ ]:
import os
import cv2
import numpy as np
import joblib
import matplotlib.pyplot as plt

from skimage.feature import local_binary_pattern, graycomatrix, graycoprops, hog

## 2. Feature extraction

Same functions as in `train_model.ipynb` so the feature vector matches what the model expects.

In [ ]:
IMG_SIZE = 224

def resize_img(img):
    return cv2.resize(img, (IMG_SIZE, IMG_SIZE))

def color_hist(img):
    feats = []
    for c in range(3):
        h = cv2.calcHist([img], [c], None, [16], [0, 256]).flatten()
        h = h / (h.sum() + 1e-7)
        feats.append(h)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    ranges = [(0, 180), (0, 256), (0, 256)]
    for c, r in enumerate(ranges):
        h = cv2.calcHist([hsv], [c], None, [16], list(r)).flatten()
        h = h / (h.sum() + 1e-7)
        feats.append(h)
    return np.concatenate(feats)

def lbp_hist(gray):
    lbp = local_binary_pattern(gray, 24, 3, method="uniform")
    hist, _ = np.histogram(lbp.ravel(), bins=26, range=(0, 26))
    hist = hist.astype("float32")
    hist /= (hist.sum() + 1e-7)
    return hist

def glcm_feat(gray):
    g = (gray // 32).astype(np.uint8)
    glcm = graycomatrix(
        g, distances=[1, 2], angles=[0, np.pi / 4, np.pi / 2, 3 * np.pi / 4],
        levels=8, symmetric=True, normed=True,
    )
    out = []
    for p in ["contrast", "dissimilarity", "homogeneity", "energy", "correlation"]:
        out.append(graycoprops(glcm, p).flatten())
    return np.concatenate(out).astype("float32")

def hog_feat(gray):
    return hog(
        gray, orientations=9, pixels_per_cell=(16, 16),
        cells_per_block=(2, 2), block_norm="L2-Hys", feature_vector=True,
    ).astype("float32")

def extract_features(img):
    img = resize_img(img)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return np.concatenate([
        color_hist(img),
        lbp_hist(gray),
        glcm_feat(gray),
        hog_feat(gray),
    ]).astype("float32")

## 3. Load the trained model

In [ ]:
MODELS_DIR = "models"

svm = joblib.load(os.path.join(MODELS_DIR, "svm_model.joblib"))
scaler = joblib.load(os.path.join(MODELS_DIR, "scaler.joblib"))
le = joblib.load(os.path.join(MODELS_DIR, "label_encoder.joblib"))

print("Classes:", list(le.classes_))

## 4. Predict on one image

Change `test_path` to try a different image.

In [ ]:
test_path = "paper/paper1.jpg"

img = cv2.imread(test_path)
if img is None:
    raise ValueError("Cannot read image: " + test_path)

feats = extract_features(img).reshape(1, -1)
feats_s = scaler.transform(feats)
probs = svm.predict_proba(feats_s)[0]
pred_idx = int(np.argmax(probs))
pred_label = str(le.classes_[pred_idx])

print("Prediction:", pred_label)
print("Confidence:", round(float(probs[pred_idx]) * 100, 2), "%")
print("\nAll probabilities:")
for i, c in enumerate(le.classes_):
    print(" ", c, "->", round(float(probs[i]) * 100, 2), "%")

rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(5, 5))
plt.imshow(rgb)
plt.title("Prediction: " + pred_label)
plt.axis("off")
plt.show()